# PersonaPlex — RunPod RTX 5090 Deployment Notebook

This notebook deploys **[PersonaPlex](https://github.com/NVIDIA/personaplex)** — NVIDIA's real-time, full-duplex
speech-to-speech model with persona/voice control (built on [Moshi](https://arxiv.org/abs/2410.00037)) — on a
**fresh RunPod pod with an RTX 5090 GPU**.

It is derived directly from the repository's own `README.md`, `client/README.md`, `moshi/pyproject.toml`,
`moshi/requirements.txt`, and the server/offline entrypoints (`moshi/moshi/server.py`, `moshi/moshi/offline.py`,
`moshi/moshi/utils/connection.py`, `moshi/moshi/models/loaders.py`). No steps are invented — every command below
maps to something the repo actually does or documents.

## What this notebook does

1. Verifies the RunPod environment, GPU and CUDA.
2. Sets up persistent storage on the RunPod volume.
3. Installs system + Python dependencies (including the **Blackwell/RTX 5090‑specific PyTorch build**).
4. Clones the repository (skipped automatically if you already uploaded it).
5. Configures Hugging Face authentication and downloads the model weights, tokenizer, voices and web UI assets.
6. Starts the PersonaPlex backend server (which also serves the prebuilt web UI — no separate frontend build is
   required for normal use).
7. Verifies the deployment with an automated offline inference smoke test and an HTTP check.
8. Documents GPU optimization notes and troubleshooting steps.

## What PersonaPlex does *not* have (so these checklist items are intentionally empty)

- **No database** of any kind — there is nothing to initialize.
- **No separate "API server"** — the single aiohttp server in `moshi/moshi/server.py` exposes both the WebSocket
  endpoint (`/api/chat`) and the static web UI on one port.
- **No Gradio/Streamlit app** — the only Gradio dependency is `gradio.networking.setup_tunnel`, an *optional*
  public-URL tunnel (`--gradio-tunnel`), not a Gradio UI. RunPod's own HTTP port proxy makes this unnecessary here.
- **No required "configuration file" to generate** — runtime behavior is controlled entirely by CLI flags and the
  HF-hosted `config.json` that ships with the model weights.

## Things this notebook *cannot* automate (you must do these yourself)

1. **Accept the NVIDIA Open Model License** for the gated model repo
   [`nvidia/personaplex-7b-v1`](https://huggingface.co/nvidia/personaplex-7b-v1) — log into Hugging Face in a
   browser and click "Agree and access repository". No script can click this for you.
2. **Create a Hugging Face access token** for that same account and have it ready to paste into the auth cell
   below.
3. **Expose the server port (default `8998`) as an HTTP Service** from the RunPod pod's *Connect* page (RunPod
   console action) so the proxy URL works from outside the pod.
4. **Grant microphone permission** in your browser when you open the live web UI — this is a per-user browser
   security prompt.

Run the cells **top to bottom**. The only cell you must intentionally run out of the normal flow is the final
**"Stop the server"** cleanup cell — leave it for whenever you actually want to shut the server down.

## 1. Environment sanity checks

Confirms this is a Linux pod with a GPU attached and a supported Python version (`moshi-personaplex` requires
Python ≥ 3.10, per `moshi/pyproject.toml`).

In [ ]:
import platform
import sys

print("Platform:", platform.platform())
print("Python:", sys.version)

assert sys.version_info >= (3, 10), (
    f"PersonaPlex (moshi/pyproject.toml) requires Python >= 3.10, found {sys.version_info}."
)
print("Python version OK.")

In [ ]:
# Confirm the NVIDIA driver sees a GPU at the OS level before we install anything.
!nvidia-smi

## 2. Persistent storage setup (RunPod volume)

RunPod mounts your persistent Network Volume at **`/workspace`**. Anything written there survives pod
stop/start (model weights are multiple GB, so re-downloading them on every restart would be wasteful).
If `/workspace` isn't present (e.g. you're running this notebook somewhere else), we fall back to the home
directory so the notebook still works end-to-end.

We also point Hugging Face's cache (`HF_HOME`) at the persistent volume so `huggingface_hub` downloads
(triggered both by this notebook and internally by `moshi/moshi/server.py` / `moshi/moshi/offline.py`) are
cached once and reused across restarts.

In [ ]:
import os

WORKSPACE = "/workspace" if os.path.isdir("/workspace") else os.path.expanduser("~")
REPO_URL = "https://github.com/MoshiHead/personaplex-original-code-streaming-prompt-with-tools-calling-v2.git"
REPO_DIR = os.path.join(WORKSPACE, "personaplex")
HF_CACHE_DIR = os.path.join(WORKSPACE, ".cache", "huggingface")
HF_REPO_ID = "nvidia/personaplex-7b-v1"   # loaders.DEFAULT_REPO in moshi/moshi/models/loaders.py

SERVER_HOST = "0.0.0.0"   # must be 0.0.0.0 (not "localhost") so RunPod's proxy can reach the server
SERVER_PORT = 8998        # default port used by moshi.server

# RunPod's HTTP port proxy terminates TLS at its edge and forwards plain HTTP to the container, so by
# default we do NOT enable the app's own self-signed TLS (--ssl). Flip this to True only if you plan to
# expose SERVER_PORT directly as a raw TCP port instead of through RunPod's HTTP proxy.
USE_APP_TLS = False

# An RTX 5090 has 32GB of VRAM, comfortably enough for this 7B model in bf16 -- CPU offload should not be
# needed. Flip to True only if you hit CUDA OOM (requires the `accelerate` package, installed below anyway).
USE_CPU_OFFLOAD = False

os.makedirs(WORKSPACE, exist_ok=True)
os.makedirs(HF_CACHE_DIR, exist_ok=True)

os.environ["HF_HOME"] = HF_CACHE_DIR
# Keep PATH aware of ~/.local/bin, where moshi/moshi/utils/connection.py installs `mkcert` if --ssl is used.
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + os.pathsep + os.environ.get("PATH", "")

print("WORKSPACE   :", WORKSPACE)
print("REPO_DIR    :", REPO_DIR)
print("HF_HOME     :", os.environ["HF_HOME"])
print("USE_APP_TLS :", USE_APP_TLS)

## 3. System package installation

Per the repo's `README.md` prerequisites: install the **Opus codec development library** before installing the
Python package (it's required by `sphn`, which PersonaPlex uses for Opus audio streaming over the WebSocket).
We also make sure `git` is present for cloning.

In [ ]:
import os

SUDO = "" if os.geteuid() == 0 else "sudo "

!{SUDO}apt-get update -qq
!{SUDO}apt-get install -y -qq --no-install-recommends git ca-certificates libopus-dev
print("System packages installed.")

## 4. Repository cloning

If `REPO_DIR` doesn't already contain the project (e.g. you uploaded your own copy of this repo to the volume
beforehand), it is cloned fresh from GitHub. If it's already there, cloning is skipped automatically — this
cell is safe to re-run on every pod restart.

In [ ]:
import pathlib
import subprocess

repo_marker = pathlib.Path(REPO_DIR) / "moshi" / "pyproject.toml"

if repo_marker.exists():
    print(f"Repository already present at {REPO_DIR}, skipping clone.")
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    print(f"Cloned into {REPO_DIR}.")

assert repo_marker.exists(), f"Expected {repo_marker} to exist after cloning/upload."


## 4b. Live web-search tool-calling for Helium

Adds the ability for the live voice assistant to answer time-sensitive questions (prices, weather,
news, sports scores, flight status, etc.) by searching the web -- **PersonaPlex's own Helium model
remains the only model in the pipeline**; no second LLM is ever called. Helium decides on its own,
via an appended system-prompt instruction, whether a question needs live information; if so it
replies with exactly `SEARCH: <query>`, and the code below intercepts that before its audio reaches
you, performs the search, and feeds the results back into the *same* Helium session (conversation
memory preserved) so it generates the real, freshly-spoken answer.

**Why this needs to patch `server.py` and `lm.py`, not just this notebook:** PersonaPlex is a
full-duplex, real-time speech model (Sections 10-14 launch `python -m moshi.server`, a live
WebSocket that generates audio and text jointly, frame by frame). There is no "send a prompt, get a
response" boundary to hook into from outside, and the client currently only sends *audio* to the
server (no text-input channel). The cells below patch the **freshly cloned repo** at `REPO_DIR`
(from Section 4) *before* it gets `pip install`-ed in Section 5, so the patched code is what actually
runs. Nothing here modifies the live web UI (`client/`) -- only how the already-running server
handles its own generated output.

**What to expect while it's running:**
- Ordinary questions get no perceptible extra delay and never trigger a search (check the server log
  to confirm).
- A live-info question causes a short **silent pause** (no filler audio) while the search runs and
  results are fed back to Helium -- typically a few seconds: the network fetch plus a bounded amount
  of time spent force-feeding the grounding text into Helium's context (capped by
  `PERSONAPLEX_SEARCH_PROMPT_MAX_CHARS`, since every character costs real GPU time to inject).
- Each search round also spends a bit of Helium's ~4-minute attention-window budget -- the same
  caveat Section 13b below already describes for the static "grounded responses" text prompt.
- Deciding when the model has *started a new utterance* (as opposed to a brief pause between words
  of the same utterance) is a heuristic, not a hard signal from the protocol -- see the
  `PERSONAPLEX_UTTERANCE_BOUNDARY_SILENCE_FRAMES` comment inside the `server.py` patch below if you
  ever see it mistime under real use. A mistimed *false arm* only costs a little extra buffering
  latency (harmless -- the buffered audio is still delivered intact); it cannot drop real speech.

Toggle the whole feature off with `ENABLE_WEB_SEARCH = False` below -- the patched server then takes
an early-exit path back to byte-for-byte original behavior.

In [ ]:
# --- Section 4b config: web-search tool-calling ----------------------------------------
# ENABLE_WEB_SEARCH=False fully reverts the server to its original behavior (the patched
# handle_agent_frame() below takes an early-exit path when this is off).
ENABLE_WEB_SEARCH = True

# "duckduckgo" is free and needs no signup. "tavily" gives higher-quality, LLM-oriented
# snippets but requires a free API key from https://tavily.com.
SEARCH_PROVIDER = "duckduckgo"   # or "tavily"

os.environ["PERSONAPLEX_ENABLE_WEB_SEARCH"] = "1" if ENABLE_WEB_SEARCH else "0"
os.environ["PERSONAPLEX_SEARCH_PROVIDER"] = SEARCH_PROVIDER

if SEARCH_PROVIDER == "tavily" and not os.environ.get("TAVILY_API_KEY"):
    from getpass import getpass
    os.environ["TAVILY_API_KEY"] = getpass("Enter your Tavily API key (input hidden): ")

print(f"ENABLE_WEB_SEARCH = {ENABLE_WEB_SEARCH}")
print(f"SEARCH_PROVIDER   = {SEARCH_PROVIDER}")


In [ ]:
import pathlib

# search_tools.py is the provider-agnostic web search adapter used by the patched
# server.py below. The rest of the pipeline only ever calls search_web(query); adding a
# new provider means writing one _search_<name>() function in this file and registering
# it -- nothing else changes. See the module docstring for details.
SEARCH_TOOLS_SOURCE = '# SPDX-License-Identifier: MIT\n"""\nProvider-agnostic web search adapter for PersonaPlex\'s live tool-calling feature.\n\nThe rest of the pipeline only ever calls `search_web(query)`. To add a new provider,\nwrite a `_search_<name>(query, max_results)` adapter that returns the same shape\n(list of {"title", "snippet", "url"} dicts) and register it in `_PROVIDERS` -- no\nother code changes. Switch providers by setting PERSONAPLEX_SEARCH_PROVIDER (or\npassing `provider=` explicitly).\n\nHelium (the PersonaPlex LM) remains the only model that reasons about the user\'s\nquestion or writes any part of the final answer -- this module only fetches raw\nsearch snippets, it never calls another LLM.\n"""\n\nimport os\nimport re\nfrom typing import Dict, List, Optional\n\nSEARCH_PROVIDER = os.environ.get("PERSONAPLEX_SEARCH_PROVIDER", "duckduckgo")\nSEARCH_MAX_RESULTS = int(os.environ.get("PERSONAPLEX_SEARCH_MAX_RESULTS", "4"))\n# Keeps each mid-conversation injection bounded: forcing text into Helium\'s own text\n# channel costs ~1/12.5s of real GPU time per character-token, so a long prompt here\n# directly extends the silent pause the user hears during a search.\nSEARCH_PROMPT_MAX_CHARS = int(os.environ.get("PERSONAPLEX_SEARCH_PROMPT_MAX_CHARS", "500"))\n\n_SEARCH_COMMAND_RE = re.compile(r"^\\s*SEARCH:\\s*(.+?)\\s*$", re.IGNORECASE)\n\nTOOL_INSTRUCTIONS = (\n    "You have a search tool. If -- and only if -- answering requires current information "\n    "you would not reliably know on your own (examples: prices, exchange rates, weather, "\n    "news, sports scores, stock prices, flight status, or anything else time-sensitive), "\n    "reply with ONLY this exact format and absolutely nothing else: SEARCH: <short search "\n    "query>. No greeting, no explanation, no extra words, no punctuation besides the query "\n    "itself. For every other question, answer normally and naturally from your own knowledge."\n)\n\n\ndef build_tool_system_prompt(role_prompt: str) -> str:\n    """Append the tool-use instructions to whatever role prompt the user configured."""\n    role_prompt = (role_prompt or "").strip()\n    if role_prompt:\n        return f"{role_prompt} {TOOL_INSTRUCTIONS}"\n    return TOOL_INSTRUCTIONS\n\n\ndef needs_search_response(text: str) -> Optional[str]:\n    """Return the query if `text` is a `SEARCH: <query>` command, else None."""\n    match = _SEARCH_COMMAND_RE.match((text or "").strip())\n    if not match:\n        return None\n    query = match.group(1).strip()\n    return query or None\n\n\ndef _get_ddgs_client():\n    # The PyPI package was renamed from duckduckgo-search to ddgs in 2025; support both\n    # so this keeps working regardless of which one ends up installed.\n    try:\n        from ddgs import DDGS\n    except ImportError:\n        from duckduckgo_search import DDGS  # type: ignore\n    return DDGS\n\n\ndef _search_duckduckgo(query: str, max_results: int) -> List[Dict[str, str]]:\n    DDGS = _get_ddgs_client()\n    with DDGS() as ddgs:\n        raw = list(ddgs.text(query, max_results=max_results))\n    return [\n        {"title": r.get("title", ""), "snippet": r.get("body", ""), "url": r.get("href", "")}\n        for r in raw\n    ]\n\n\ndef _search_tavily(query: str, max_results: int) -> List[Dict[str, str]]:\n    from tavily import TavilyClient\n\n    client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])\n    response = client.search(query, max_results=max_results)\n    return [\n        {"title": r.get("title", ""), "snippet": r.get("content", ""), "url": r.get("url", "")}\n        for r in response.get("results", [])\n    ]\n\n\n_PROVIDERS = {\n    "duckduckgo": _search_duckduckgo,\n    "tavily": _search_tavily,\n}\n\n\ndef search_web(\n    query: str,\n    max_results: Optional[int] = None,\n    provider: Optional[str] = None,\n) -> List[Dict[str, str]]:\n    """Provider-agnostic web search. Returns [] / an explanatory pseudo-result on failure\n    rather than raising, since callers run this mid-conversation and must always be able\n    to build *some* follow-up prompt for Helium."""\n    provider = provider or SEARCH_PROVIDER\n    max_results = max_results or SEARCH_MAX_RESULTS\n    adapter = _PROVIDERS.get(provider)\n    if adapter is None:\n        raise ValueError(f"Unknown search provider \'{provider}\'. Available: {list(_PROVIDERS)}")\n    try:\n        return adapter(query, max_results)\n    except Exception as exc:\n        return [{"title": "", "snippet": f"(search failed: {exc})", "url": ""}]\n\n\ndef build_search_prompt(query: str, results: List[Dict[str, str]]) -> str:\n    """Build the grounding text force-fed back into Helium after a search, reusing the\n    same \'<system> ... Information: ...\' pattern the notebook already documents for\n    static topic grounding (Section 13b) -- that phrasing is in-distribution for the\n    model, so it follows it more reliably than an arbitrary instruction."""\n    snippets = [r.get("snippet", "").strip() for r in results]\n    if not results or not any(snippets):\n        return (\n            f"Information: a search for \'{query}\' returned no useful results. Tell the user "\n            f"you could not find current information on this."\n        )\n    lines = []\n    for r in results:\n        label = r.get("title") or r.get("url") or "result"\n        snippet = r.get("snippet", "").strip()\n        if snippet:\n            lines.append(f"{label}: {snippet}")\n    body = " | ".join(lines)\n    prompt = (\n        f"Information: search results for \'{query}\': {body} Using ONLY the information "\n        f"above, answer the user\'s question naturally and conversationally, as if you "\n        f"already knew it. If it is insufficient, say so."\n    )\n    return prompt[:SEARCH_PROMPT_MAX_CHARS]\n'

search_tools_path = pathlib.Path(REPO_DIR) / "moshi" / "moshi" / "search_tools.py"
search_tools_path.write_text(SEARCH_TOOLS_SOURCE, encoding="utf-8")
print(f"Wrote {search_tools_path} ({len(SEARCH_TOOLS_SOURCE)} chars)")


In [ ]:
%pip install -q ddgs tavily-python

In [ ]:
import importlib.util
import pathlib

# Fail fast on network/dependency issues before spending time loading the (multi-GB)
# model in Section 8 -- same "verify early" spirit as Sections 12/13's smoke tests.
_search_tools_path = pathlib.Path(REPO_DIR) / "moshi" / "moshi" / "search_tools.py"
_spec = importlib.util.spec_from_file_location("search_tools_verify", _search_tools_path)
_search_tools_verify = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_search_tools_verify)

_results = _search_tools_verify.search_web("what is today's date")
assert _results, "search_web() returned no results -- check network access and SEARCH_PROVIDER config."
print(f"Search provider '{_search_tools_verify.SEARCH_PROVIDER}' OK -- got {len(_results)} result(s):")
for r in _results[:2]:
    print(" -", r.get("title") or r.get("url"), "|", r.get("snippet", "")[:80])


In [ ]:
import pathlib

# Adds LMGen.inject_agent_text[_async] -- purely additive, no existing method is
# modified. This is the mechanism that lets server.py force search results into
# Helium's context mid-conversation without resetting streaming state (see the notebook
# markdown above and lm.py's own _step_text_prompt, which this reuses).
lm_path = pathlib.Path(REPO_DIR) / "moshi" / "moshi" / "models" / "lm.py"
_src = lm_path.read_text(encoding="utf-8")

_LM_OLD = '    async def step_system_prompts_async(self, mimi, is_alive: Optional[Callable]=None):\n        await self._step_voice_prompt_async(mimi, is_alive)\n        await self._step_audio_silence_async(is_alive)\n        await self._step_text_prompt_async(is_alive)\n        await self._step_audio_silence_async(is_alive)'

_LM_NEW = '    def inject_agent_text(self, tokens: list[int]):\n        """Force-feed new text into the agent\'s own text channel mid-conversation,\n        without resetting streaming state. Reuses the exact mechanism that loads the\n        initial system prompt (_step_text_prompt) so the LM\'s KV-cache / conversation\n        memory is preserved -- this is what makes mid-call tool-result injection possible\n        for the web-search feature (see search_tools.py and server.py\'s search_state)."""\n        self.text_prompt_tokens = tokens\n        self._step_text_prompt()\n\n    async def inject_agent_text_async(self, tokens: list[int], is_alive: Optional[Callable] = None):\n        self.text_prompt_tokens = tokens\n        await self._step_text_prompt_async(is_alive)\n\n    async def step_system_prompts_async(self, mimi, is_alive: Optional[Callable]=None):\n        await self._step_voice_prompt_async(mimi, is_alive)\n        await self._step_audio_silence_async(is_alive)\n        await self._step_text_prompt_async(is_alive)\n        await self._step_audio_silence_async(is_alive)'

if "def inject_agent_text(" in _src:
    print(f"{lm_path} already patched -- skipping.")
else:
    _count = _src.count(_LM_OLD)
    assert _count == 1, (
        f"Expected exactly 1 match for the step_system_prompts_async anchor in lm.py, found {_count}. "
        "Upstream file may have changed -- update this patch cell."
    )
    lm_path.write_text(_src.replace(_LM_OLD, _LM_NEW), encoding="utf-8")
    print(f"Patched {lm_path} -- added LMGen.inject_agent_text / inject_agent_text_async.")


In [ ]:
import pathlib

# Adds the web-search tool-calling interception to server.py's live frame loop. See the
# notebook markdown above for the full design; the short version: Helium's own "SEARCH:
# <query>" reply is caught before its audio reaches the client, search_web() runs, and
# the results are force-fed back into the same live session via LMGen.inject_agent_text_async
# (added to lm.py by the previous cell) so Helium generates the real spoken answer itself.
server_path = pathlib.Path(REPO_DIR) / "moshi" / "moshi" / "server.py"
_src = server_path.read_text(encoding="utf-8")


def _apply(text, old, new, label):
    _count = text.count(old)
    assert _count == 1, (
        f"[{label}] expected exactly 1 match while patching server.py, found {_count}. "
        "Upstream file may have changed -- update this patch cell."
    )
    return text.replace(old, new)


if "search_state" in _src:
    print(f"{server_path} already patched -- skipping.")
else:
    _OLD_IMPORTS = 'from .client_utils import make_log, colorize\nfrom .models import loaders, MimiModel, LMModel, LMGen\nfrom .utils.connection import create_ssl_context, get_lan_ip\nfrom .utils.logging import setup_logger, ColorizedLog'
    _NEW_IMPORTS = 'from .client_utils import make_log, colorize\nfrom .models import loaders, MimiModel, LMModel, LMGen\nfrom .search_tools import build_tool_system_prompt, build_search_prompt, needs_search_response, search_web\nfrom .utils.connection import create_ssl_context, get_lan_ip\nfrom .utils.logging import setup_logger, ColorizedLog\n\n\n# --- Web-search tool-calling config (see notebook Section 4b) -------------------------\n# Helium (the PersonaPlex LM) remains the only model in this pipeline; these constants\n# only control the *server-side interception* of its own "SEARCH: <query>" replies, not\n# how the model reasons. See search_tools.py for the search backend itself.\nENABLE_WEB_SEARCH = os.environ.get("PERSONAPLEX_ENABLE_WEB_SEARCH", "1") == "1"\n# There is no ground-truth "utterance boundary" signal in this full-duplex protocol --\n# PAD/EPAD text tokens occur both between agent utterances AND between word-pieces of\n# ongoing speech (text and audio are emitted on independent, delayed schedules). We treat\n# a run of at least this many consecutive PAD frames as "the agent was quiet", and only\n# arm search-detection on the speech that follows such a run. A too-short value costs a\n# little extra buffering latency on false arms (harmless -- buffered audio is still\n# flushed intact); a too-long value can miss a real "SEARCH:" utterance. Tune by observing\n# real conversations if needed.\nUTTERANCE_BOUNDARY_SILENCE_FRAMES = int(os.environ.get("PERSONAPLEX_UTTERANCE_BOUNDARY_SILENCE_FRAMES", "10"))\nSEARCH_DECISION_MAX_FRAMES = 20    # cap on frames spent deciding if an utterance is "SEARCH:"\nSEARCH_CAPTURE_SILENCE_FRAMES = 6  # ~0.5s trailing silence => the query is complete\nSEARCH_CAPTURE_MAX_CHARS = 200     # safety cap on a runaway query capture\nMAX_CONSECUTIVE_SEARCHES = 2       # loop guard: stop auto-searching after N in a row'
    _src = _apply(_src, _OLD_IMPORTS, _NEW_IMPORTS, "imports")

    _OLD_TEXT_PROMPT = '        self.lm_gen.text_prompt_tokens = self.text_tokenizer.encode(wrap_with_system_tags(request.query["text_prompt"])) if len(request.query["text_prompt"]) > 0 else None'
    _NEW_TEXT_PROMPT = '        base_text_prompt = request.query["text_prompt"]\n        combined_text_prompt = build_tool_system_prompt(base_text_prompt) if ENABLE_WEB_SEARCH else base_text_prompt\n        self.lm_gen.text_prompt_tokens = self.text_tokenizer.encode(wrap_with_system_tags(combined_text_prompt)) if len(combined_text_prompt) > 0 else None'
    _src = _apply(_src, _OLD_TEXT_PROMPT, _NEW_TEXT_PROMPT, "text_prompt")

    _OLD_CLOSE_INIT = '        close = False\n        async with self.lock:'
    _NEW_CLOSE_INIT = '        close = False\n        # Per-connection state for the web-search tool-calling interception below.\n        search_state = {\n            "mode": "live",  # live | buffering | capturing\n            "text": "",\n            "frames": [],\n            "silence_run": 0,\n            "consecutive_searches": 0,\n        }\n        async with self.lock:'
    _src = _apply(_src, _OLD_CLOSE_INIT, _NEW_CLOSE_INIT, "close_init")

    _OLD_OPUS_BLOCK = '        async def opus_loop():\n            all_pcm_data = None\n\n            while True:\n                if close:\n                    return\n                await asyncio.sleep(0.001)\n                pcm = opus_reader.read_pcm()\n                if pcm.shape[-1] == 0:\n                    continue\n                if all_pcm_data is None:\n                    all_pcm_data = pcm\n                else:\n                    all_pcm_data = np.concatenate((all_pcm_data, pcm))\n                while all_pcm_data.shape[-1] >= self.frame_size:\n                    be = time.time()\n                    chunk = all_pcm_data[: self.frame_size]\n                    all_pcm_data = all_pcm_data[self.frame_size:]\n                    chunk = torch.from_numpy(chunk)\n                    chunk = chunk.to(device=self.device)[None, None]\n                    codes = self.mimi.encode(chunk)\n                    _ = self.other_mimi.encode(chunk)\n                    for c in range(codes.shape[-1]):\n                        tokens = self.lm_gen.step(codes[:, :, c: c + 1])\n                        if tokens is None:\n                            continue\n                        assert tokens.shape[1] == self.lm_gen.lm_model.dep_q + 1\n                        main_pcm = self.mimi.decode(tokens[:, 1:9])\n                        _ = self.other_mimi.decode(tokens[:, 1:9])\n                        main_pcm = main_pcm.cpu()\n                        opus_writer.append_pcm(main_pcm[0, 0].numpy())\n                        text_token = tokens[0, 0, 0].item()\n                        if text_token not in (0, 3):\n                            _text = self.text_tokenizer.id_to_piece(text_token)  # type: ignore\n                            _text = _text.replace("▁", " ")\n                            msg = b"\\x02" + bytes(_text, encoding="utf8")\n                            await ws.send_bytes(msg)\n                        else:\n                            text_token_map = [\'EPAD\', \'BOS\', \'EOS\', \'PAD\']\n\n        async def send_loop():'
    _NEW_OPUS_BLOCK = '        async def opus_loop():\n            all_pcm_data = None\n\n            while True:\n                if close:\n                    return\n                await asyncio.sleep(0.001)\n                pcm = opus_reader.read_pcm()\n                if pcm.shape[-1] == 0:\n                    continue\n                if all_pcm_data is None:\n                    all_pcm_data = pcm\n                else:\n                    all_pcm_data = np.concatenate((all_pcm_data, pcm))\n                while all_pcm_data.shape[-1] >= self.frame_size:\n                    be = time.time()\n                    chunk = all_pcm_data[: self.frame_size]\n                    all_pcm_data = all_pcm_data[self.frame_size:]\n                    chunk = torch.from_numpy(chunk)\n                    chunk = chunk.to(device=self.device)[None, None]\n                    codes = self.mimi.encode(chunk)\n                    _ = self.other_mimi.encode(chunk)\n                    for c in range(codes.shape[-1]):\n                        tokens = self.lm_gen.step(codes[:, :, c: c + 1])\n                        if tokens is None:\n                            continue\n                        assert tokens.shape[1] == self.lm_gen.lm_model.dep_q + 1\n                        text_token = tokens[0, 0, 0].item()\n                        await handle_agent_frame(tokens, text_token)\n\n        # --- Web-search tool-calling (see notebook Section 4b) --------------------------\n        # Intercepts a "SEARCH: <query>" reply from Helium before its audio ever reaches\n        # the client, runs search_web(), and force-feeds the results back into Helium\'s\n        # own text channel (LMGen.inject_agent_text_async) so it can generate a real,\n        # freshly-sampled spoken answer -- Helium is still the only model writing any\n        # word the user hears; this code only decides *when* to hand it search results.\n        def _decode_frame(tokens):\n            pcm = self.mimi.decode(tokens[:, 1:9])\n            _ = self.other_mimi.decode(tokens[:, 1:9])\n            return pcm.cpu()[0, 0].numpy()\n\n        def _piece_for(text_token):\n            if text_token in (0, 3):\n                return None\n            piece = self.text_tokenizer.id_to_piece(text_token)  # type: ignore\n            return piece.replace("▁", " ")\n\n        async def _send_frame(pcm, piece):\n            opus_writer.append_pcm(pcm)\n            if piece is not None:\n                await ws.send_bytes(b"\\x02" + bytes(piece, encoding="utf8"))\n\n        async def flush_search_buffer():\n            for pcm, piece in search_state["frames"]:\n                await _send_frame(pcm, piece)\n            search_state["frames"] = []\n\n        async def run_search_and_inject(query: str):\n            clog.log("info", f"tool call detected: SEARCH: {query}")\n            loop = asyncio.get_event_loop()\n            results = await loop.run_in_executor(None, search_web, query)\n            prompt_text = build_search_prompt(query, results)\n            clog.log("info", f"injecting search-grounded prompt ({len(prompt_text)} chars)")\n            new_tokens = self.text_tokenizer.encode(wrap_with_system_tags(prompt_text))\n            await self.lm_gen.inject_agent_text_async(new_tokens, is_alive=is_alive)\n            # Mirrors the reset already done after the initial system prompt load below:\n            # frames were deliberately never decoded while capturing the query, so Mimi\'s\n            # own streaming decode state (not the LM\'s) needs a clean point to resume from.\n            self.mimi.reset_streaming()\n            self.other_mimi.reset_streaming()\n\n        async def handle_agent_frame(tokens, text_token):\n            if not ENABLE_WEB_SEARCH:\n                await _send_frame(_decode_frame(tokens), _piece_for(text_token))\n                return\n\n            is_silent = text_token in (0, 3)\n            mode = search_state["mode"]\n\n            if mode == "live":\n                if is_silent:\n                    search_state["silence_run"] += 1\n                    await _send_frame(_decode_frame(tokens), None)\n                    return\n                # A real text piece follows this frame. Only treat it as the start of a\n                # fresh agent utterance (and arm search-detection) if it follows a\n                # sufficiently long pause -- a single PAD frame between word-pieces of\n                # ongoing speech must not re-arm detection mid-sentence.\n                if (search_state["silence_run"] >= UTTERANCE_BOUNDARY_SILENCE_FRAMES\n                        and search_state["consecutive_searches"] < MAX_CONSECUTIVE_SEARCHES):\n                    search_state["mode"] = "buffering"\n                    search_state["text"] = ""\n                    search_state["frames"] = []\n                    search_state["silence_run"] = 0\n                    mode = "buffering"\n                else:\n                    search_state["silence_run"] = 0\n                    await _send_frame(_decode_frame(tokens), _piece_for(text_token))\n                    return\n\n            if mode == "buffering":\n                piece = _piece_for(text_token)\n                search_state["frames"].append((_decode_frame(tokens), piece))\n                if piece:\n                    search_state["text"] += piece\n                stripped = search_state["text"].strip().upper()\n                target = "SEARCH:"\n                if stripped.startswith(target):\n                    search_state["mode"] = "capturing"\n                    search_state["frames"] = []  # this audio is never played\n                elif target.startswith(stripped) and len(search_state["frames"]) < SEARCH_DECISION_MAX_FRAMES:\n                    pass  # still ambiguous (e.g. "SE"); keep buffering, bounded by the cap above\n                else:\n                    await flush_search_buffer()\n                    search_state["mode"] = "live"\n                    search_state["consecutive_searches"] = 0\n                return\n\n            if mode == "capturing":\n                # Audio for this utterance is never decoded or sent -- satisfies the\n                # "hard pause, no filler" behavior while the query is captured/searched.\n                if is_silent:\n                    search_state["silence_run"] += 1\n                else:\n                    search_state["silence_run"] = 0\n                    piece = _piece_for(text_token)\n                    if piece:\n                        search_state["text"] += piece\n                done = (\n                    search_state["silence_run"] >= SEARCH_CAPTURE_SILENCE_FRAMES\n                    or len(search_state["text"]) >= SEARCH_CAPTURE_MAX_CHARS\n                )\n                if done:\n                    query = needs_search_response(search_state["text"])\n                    search_state["mode"] = "live"\n                    search_state["text"] = ""\n                    search_state["silence_run"] = 0\n                    search_state["consecutive_searches"] += 1\n                    if query:\n                        await run_search_and_inject(query)\n                return\n\n        async def send_loop():'
    _src = _apply(_src, _OLD_OPUS_BLOCK, _NEW_OPUS_BLOCK, "opus_block")

    server_path.write_text(_src, encoding="utf-8")
    print(f"Patched {server_path} -- added the web-search tool-calling state machine.")


## 5. Python dependency installation

The repo's documented install command is:

```bash
pip install moshi/.
```

which installs from `moshi/pyproject.toml` (`numpy`, `safetensors`, `huggingface-hub`, `einops`,
`sentencepiece`, `sounddevice`, `sphn`, `torch>=2.2,<2.5`, `aiohttp`).

### RTX 5090 / Blackwell note

The pinned `torch<2.5` build does **not** ship CUDA kernels for Blackwell (RTX 50‑series, `sm_120`) GPUs. The
repo's `README.md` explicitly documents the fix for this
([NVIDIA/personaplex#2](https://github.com/NVIDIA/personaplex/issues/2)): reinstall PyTorch from the `cu130`
wheel index *after* the base install. This intentionally overrides the `<2.5` pin — that's expected and is the
upstream-recommended fix, not a mistake.

In [ ]:
%pip install -q --upgrade pip setuptools wheel
%pip install -q "{REPO_DIR}/moshi/." 

In [ ]:
# Blackwell (RTX 5090) requires CUDA-13.0-built PyTorch wheels. This intentionally supersedes the
# torch<2.5 pin from moshi/pyproject.toml -- see README.md "Extra step for Blackwell based GPUs".
%pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu130

In [ ]:
# `accelerate` enables --cpu-offload (README's "CPU Offload" section); `gradio` enables the optional
# --gradio-tunnel fallback for public exposure if you ever need it instead of RunPod's HTTP proxy.
%pip install -q accelerate gradio

## 6. CUDA / GPU verification

Confirms PyTorch can see the RTX 5090 and that the installed build actually has working CUDA kernels for it
(a bf16 matmul smoke test) — this is the check that would have failed before the `cu130` reinstall above if it
had been skipped.

In [ ]:
import torch

print("Torch version      :", torch.__version__)
print("Torch CUDA version :", torch.version.cuda)
print("CUDA available     :", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU detected by PyTorch. Confirm this RunPod pod has an RTX 5090 GPU attached "
        "and that the driver is healthy (see the `nvidia-smi` output above)."
    )

device_name = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)
print("GPU                :", device_name)
print("Compute capability :", capability)

if "5090" not in device_name and "RTX 50" not in device_name:
    print(f"WARNING: expected an RTX 5090, found '{device_name}'. Continuing anyway.")

# Smoke test: this is exactly the kind of op that fails with
# "no kernel image is available for execution on the device" on Blackwell if torch wasn't
# reinstalled from the cu130 index.
x = torch.randn(4096, 4096, device="cuda", dtype=torch.bfloat16)
y = x @ x
torch.cuda.synchronize()
print("bf16 CUDA matmul smoke test OK, result shape:", tuple(y.shape))

## 7. Hugging Face authentication

**Manual step required before running this cell:** log into Hugging Face in a browser, open
[`nvidia/personaplex-7b-v1`](https://huggingface.co/nvidia/personaplex-7b-v1), and click **"Agree and access
repository"** to accept the NVIDIA Open Model License. Then create an access token (read access is enough) at
<https://huggingface.co/settings/tokens>.

The repo's `README.md` documents this as `export HF_TOKEN=<YOUR_HUGGINGFACE_TOKEN>`; the cell below does the
same thing from inside the notebook, via a hidden prompt so the token isn't echoed into cell output.

In [ ]:
from getpass import getpass

from huggingface_hub import login

# hf_token = os.environ.get("HF_TOKEN")
# sa token...not mine
hf_token = 'tLNSyNjFduNaLUbvyxosVqiGwuAtiQPOTt'    
if not hf_token:
    hf_token = getpass("Enter your Hugging Face access token (input hidden): ")

os.environ["HF_TOKEN"] = hf_token
login(token=hf_token, add_to_git_credential=False)
print("Logged in to Hugging Face Hub.")

## 8. Model downloading

`moshi/moshi/server.py` and `moshi/moshi/offline.py` both lazily call `hf_hub_download` for each asset the
first time they need it. We pre-fetch the same files here so that (a) any license/token problem surfaces now
with a clear error instead of mid-startup, and (b) everything is already warm in the `HF_HOME` cache before we
launch the server.

Assets (from `moshi/moshi/models/loaders.py` and `moshi/moshi/server.py`):
- `config.json` — model config
- `tokenizer_spm_32k_3.model` — SentencePiece text tokenizer
- `tokenizer-e351c8d8-checkpoint125.safetensors` — Mimi codec weights
- `model.safetensors` — Moshi/PersonaPlex LM weights
- `voices.tgz` — packaged voice-prompt embeddings (NATF0‑3, NATM0‑3, VARF0‑4, VARM0‑4)
- `dist.tgz` — the **prebuilt web UI** (this is why no separate frontend build step is needed)

In [ ]:
import tarfile

from huggingface_hub import hf_hub_download

ASSET_FILES = [
    "config.json",
    "tokenizer_spm_32k_3.model",
    "tokenizer-e351c8d8-checkpoint125.safetensors",
    "model.safetensors",
    "voices.tgz",
    "dist.tgz",
]

downloaded = {}
try:
    for fname in ASSET_FILES:
        # No explicit cache_dir: this honors HF_HOME (set in step 2) so the notebook and the server
        # subprocess we launch later (which inherits the same env) share one cache.
        path = hf_hub_download(HF_REPO_ID, fname)
        downloaded[fname] = path
        print(f"OK  {fname} -> {path}")
except Exception as e:
    raise RuntimeError(
        "Failed to download model assets from "
        f"https://huggingface.co/{HF_REPO_ID}. This almost always means either:\n"
        "  1) you have not clicked 'Agree and access repository' on that model page yet, or\n"
        "  2) the HF_TOKEN you supplied doesn't belong to the account that accepted the license, or\n"
        "  3) the token is invalid/expired.\n"
        f"Original error: {e}"
    )

In [ ]:
import pathlib

# Pre-extract the tarballs once, exactly like _get_voice_prompt_dir / _get_static_path do in
# moshi/moshi/server.py, so the first real request doesn't pay the extraction cost.
for tgz_name in ("voices.tgz", "dist.tgz"):
    tgz_path = pathlib.Path(downloaded[tgz_name])
    out_dir = tgz_path.parent / tgz_name.replace(".tgz", "")
    if not out_dir.exists():
        with tarfile.open(tgz_path, "r:gz") as tar:
            tar.extractall(path=tgz_path.parent)
    print(f"{tgz_name} -> {out_dir} ({'already extracted' if out_dir.exists() else 'extracted now'})")

## 9. TLS certificate directory (only used if `USE_APP_TLS = True`)

The README's default local-machine workflow is:

```bash
SSL_DIR=$(mktemp -d); python -m moshi.server --ssl "$SSL_DIR"
```

`--ssl` makes `moshi/moshi/utils/connection.py` auto-install `mkcert` and generate a **self-signed**
certificate in that directory. That's appropriate for direct LAN/local access, but on RunPod we're putting the
server behind RunPod's own HTTPS proxy (Section 11), which already terminates TLS at the edge — adding a second,
self-signed TLS layer underneath it would just produce certificate warnings for no benefit. We still create the
directory here so you can flip `USE_APP_TLS = True` above if you'd rather expose `SERVER_PORT` as a raw TCP port
instead of through the HTTP proxy.

In [ ]:
import tempfile

SSL_DIR = tempfile.mkdtemp(prefix="personaplex_ssl_")
print("SSL_DIR:", SSL_DIR, "(only used if USE_APP_TLS = True)")

## 9b. (Optional) Build the web client from source — raise the Text Prompt limit

By default `python -m moshi.server` does **not** use the `client/` source in this repo at all: when
`--static` is omitted, `_get_static_path` in `moshi/moshi/server.py` downloads NVIDIA's **prebuilt** web UI
bundle (`dist.tgz`, extracted in Section 8) and serves that instead. That prebuilt bundle caps the "Text
Prompt" box at 1000 characters (`client/src/pages/Queue/Queue.tsx`).

If you plan to give the model a longer topic/grounding prompt (see Section 13b below), run the two cells
below to:
1. Install Node.js, matching `client/.nvmrc`.
2. Patch the cloned repo's `Queue.tsx` to raise that limit to `TEXT_PROMPT_MAX_LEN` characters and add a
   "Topic-only (grounded)" preset.
3. Build the client (`npm ci && npm run build`, per `client/README.md`) and record the build output so
   Section 10 can launch the server with `--static` pointing at it instead of the prebuilt bundle.

Skip this section if a prompt under 1000 characters is enough — the server falls back to NVIDIA's prebuilt
UI automatically when `STATIC_DIR` is left unset.

In [ ]:
import shutil
import subprocess

NODE_MAJOR = "20"  # matches client/.nvmrc (v20.12.2)

def _node_major_ok():
    node = shutil.which("node")
    if not node:
        return False
    out = subprocess.run([node, "-v"], capture_output=True, text=True).stdout.strip()
    return out.startswith(f"v{NODE_MAJOR}.")

if _node_major_ok():
    print("Node.js already installed:", subprocess.run(["node", "-v"], capture_output=True, text=True).stdout.strip())
else:
    !{SUDO}apt-get install -y -qq curl
    !curl -fsSL https://deb.nodesource.com/setup_{NODE_MAJOR}.x | {SUDO}bash -
    !{SUDO}apt-get install -y -qq nodejs
    print("Node.js installed:", subprocess.run(["node", "-v"], capture_output=True, text=True).stdout.strip())


In [ ]:
import os
import pathlib
import subprocess

TEXT_PROMPT_MAX_LEN = 4000  # raise/lower as needed — see the context-window note in Section 13b below

queue_tsx = pathlib.Path(REPO_DIR) / "client" / "src" / "pages" / "Queue" / "Queue.tsx"
src = queue_tsx.read_text(encoding="utf-8")
patched = src.replace("maxLength={1000}", f"maxLength={{{TEXT_PROMPT_MAX_LEN}}}")
patched = patched.replace("{textPrompt.length}/1000", f"{{textPrompt.length}}/{TEXT_PROMPT_MAX_LEN}")
if patched != src:
    queue_tsx.write_text(patched, encoding="utf-8")
    print(f"Patched {queue_tsx} -> Text Prompt limit is now {TEXT_PROMPT_MAX_LEN} characters.")
else:
    print(f"{queue_tsx}: pattern not found (already patched, or upstream file changed) — left unchanged.")

STATIC_DIR = None
client_dir = os.path.join(REPO_DIR, "client")
try:
    subprocess.run(["npm", "ci"], cwd=client_dir, check=True)
    subprocess.run(["npm", "run", "build"], cwd=client_dir, check=True)
    built_dist = os.path.join(client_dir, "dist")
    assert os.path.isdir(built_dist), f"Expected {built_dist} to exist after `npm run build`."
    STATIC_DIR = built_dist
    print("Custom client built at:", STATIC_DIR)
except Exception as e:
    print(f"Client build failed ({e!r}); falling back to NVIDIA's prebuilt UI (1000-char Text Prompt limit).")
    STATIC_DIR = None


## 10. Backend (+ web UI) startup

This is the repo's **recommended and only documented launch method** — `python -m moshi.server`. It is a
single aiohttp process that serves:
- the WebSocket endpoint `/api/chat` (the real-time speech protocol), and
- the web UI at `/` — either the prebuilt `dist.tgz` bundle (downloaded in Section 8), or your own
  client build from Section 9b if `STATIC_DIR` was set there (passed to the server via `--static`).
  Either way there is **no separate frontend process to start** for normal use.

A notebook cell that calls `web.run_app(...)` directly would block the kernel forever, so we launch it as a
background subprocess and log to a file, then poll that log in the next cell.

In [ ]:
import subprocess
import sys

env = os.environ.copy()
LOG_PATH = os.path.join(WORKSPACE, "personaplex_server.log")

cmd = [sys.executable, "-m", "moshi.server", "--host", SERVER_HOST, "--port", str(SERVER_PORT)]
if USE_APP_TLS:
    cmd += ["--ssl", SSL_DIR]
if USE_CPU_OFFLOAD:
    cmd += ["--cpu-offload"]
if globals().get("STATIC_DIR"):
    cmd += ["--static", STATIC_DIR]

print("Launching:", " ".join(cmd))

log_file = open(LOG_PATH, "w")
server_proc = subprocess.Popen(
    cmd, cwd=os.path.join(REPO_DIR, "moshi"), env=env, stdout=log_file, stderr=subprocess.STDOUT,
)
print(f"Server launched with PID {server_proc.pid}. Logs: {LOG_PATH}")

In [ ]:
import time

def tail(path, n=60):
    with open(path) as f:
        return "".join(f.readlines()[-n:])

READY_MARKER = "Access the Web UI directly at"
TIMEOUT_S = 900  # first run downloads + loads a multi-GB model; be generous
POLL_S = 5

start = time.time()
ready = False
while time.time() - start < TIMEOUT_S:
    if server_proc.poll() is not None:
        print(tail(LOG_PATH))
        raise RuntimeError(
            f"Server process exited early with return code {server_proc.returncode}. See log above."
        )
    if READY_MARKER in open(LOG_PATH).read():
        ready = True
        break
    time.sleep(POLL_S)

print(tail(LOG_PATH))

if not ready:
    raise TimeoutError(
        f"Server did not report readiness within {TIMEOUT_S}s. Check the log tail above -- "
        "the most common causes are a slow first-time model download or a CUDA/driver mismatch."
    )

print("\nServer is up and warmed up.")

## 11. Expose the port & get the access URL

RunPod will not route external traffic to a port unless you explicitly expose it. In the pod's **Connect**
page, add an **HTTP Service** for `SERVER_PORT` (default `8998`) if you haven't already. RunPod then serves it
at `https://<POD_ID>-<PORT>.proxy.runpod.net`, with RunPod terminating TLS — which is exactly why
`USE_APP_TLS = False` is the right default here (see Section 9).

In [ ]:
pod_id = os.environ.get("RUNPOD_POD_ID")

if pod_id:
    public_url = f"https://{pod_id}-{SERVER_PORT}.proxy.runpod.net"
    print("RunPod public URL (requires SERVER_PORT to be exposed as an HTTP Service on the pod's Connect page):")
    print(" ", public_url)
else:
    print(
        "RUNPOD_POD_ID was not found in the environment. If you are on RunPod, expose "
        f"port {SERVER_PORT} as an HTTP Service from the pod's Connect page and use the proxy URL shown there."
    )

scheme = "https" if USE_APP_TLS else "http"
print(f"Local URL inside the pod: {scheme}://localhost:{SERVER_PORT}")

## 12. Verification — offline inference smoke test

This runs the repo's documented `moshi.offline` "Assistant example" exactly as given in `README.md` — it
streams `assets/test/input_assistant.wav` through the model with voice prompt `NATF2.pt` and writes an output
WAV + JSON transcript. This doesn't need a browser, microphone, or open port, so it's a good way to confirm the
backend, weights and GPU are all working correctly before testing the live UI.

In [ ]:
import subprocess
import sys

offline_cmd = [
    sys.executable, "-m", "moshi.offline",
    "--voice-prompt", "NATF2.pt",
    "--input-wav", "assets/test/input_assistant.wav",
    "--seed", "42424242",
    "--output-wav", "output_assistant.wav",
    "--output-text", "output_assistant.json",
]

result = subprocess.run(
    offline_cmd, cwd=REPO_DIR, env=os.environ.copy(), capture_output=True, text=True,
)

print(result.stdout[-4000:])
if result.returncode != 0:
    print(result.stderr[-4000:])
    raise RuntimeError("Offline smoke test failed -- see output above.")

print("\nOffline smoke test succeeded.")

In [ ]:
import json

from IPython.display import Audio, display

output_wav_path = os.path.join(REPO_DIR, "output_assistant.wav")
output_json_path = os.path.join(REPO_DIR, "output_assistant.json")

display(Audio(output_wav_path))

with open(output_json_path) as f:
    tokens = json.load(f)
print("Generated text tokens (joined):")
print("".join(tokens))

## 13. Verification — HTTP check on the live server

Confirms the running server process answers on `/` (the web UI's `index.html`, served from `dist.tgz`).

In [ ]:
import ssl
import urllib.request

scheme = "https" if USE_APP_TLS else "http"
ctx = ssl._create_unverified_context() if USE_APP_TLS else None

with urllib.request.urlopen(f"{scheme}://localhost:{SERVER_PORT}/", context=ctx, timeout=15) as resp:
    print("HTTP status :", resp.status)
    print("Content-Type:", resp.headers.get("Content-Type"))
    assert resp.status == 200, f"Expected 200, got {resp.status}"

print("Web UI is being served correctly.")

## 13b. Provide topic information for grounded responses

PersonaPlex exposes exactly one channel for "give it information, then make it stick only to that": the
**Text Prompt** field already in the web UI. That field *is* the model's system prompt — `server.py` wraps
whatever you type in `<system> ... <system>` tags (`wrap_with_system_tags`) and feeds it to the model token
by token as the very first thing in its context, before either of you say a word
(`moshi/moshi/models/lm.py: step_system_prompts`). There is no separate "context" field or knowledge base —
everything goes through this one box.

**Where to put it:** type/paste it into the Text Prompt box in Section 14 below, *before* you click connect.
Anything you only say out loud after connecting is ordinary spoken conversation, not a system prompt — it
is not privileged the way text typed into that box is.

**How to phrase it:** PersonaPlex's own training data already has a pattern for exactly this — a short
role sentence followed by `Information: <facts>` (see the "Customer Service Roles" examples in
`README.md`). That pattern is in-distribution for the model, so it follows it more reliably than a generic
"only answer from this text" instruction wrapped around a long free-form document. The **"Topic-only
(grounded)"** preset added to the web UI (`client/src/pages/Queue/Queue.tsx`, only present if you built the
client in Section 9b) gives you a starting template — replace its bracketed placeholders with your topic
and facts. The cell below builds and prints the same kind of string for you to copy in.

**Limits this does not remove:**
1. This is a soft instruction a 7B model tries to follow, not a hard filter. There is no
   retrieval/grounding-enforcement layer in this codebase, so it can still drift, guess, or blend in outside
   knowledge — "strict" here is best-effort prompting, not a guarantee.
2. The model's causal attention only spans the last `context = 3000` steps
   (`moshi/moshi/models/loaders.py`) at `FRAME_RATE = 12.5` Hz — about 4 minutes of combined
   prompt-loading + silence + conversation. Every character of your Text Prompt is loaded as its own step
   before the conversation starts and counts against that same 4-minute budget: a longer prompt means a
   longer pause before you can start talking, *and* less of the window left before the model can no longer
   attend back to your original prompt at all (it falls out of the attention window like anything else
   `context` steps in the past). There is no mechanism in this code to re-inject the prompt mid-conversation
   — for sessions longer than a few minutes, treat the grounding as something that fades, not something
   permanent.
3. Given (2), prefer a tight list of concrete facts (closer to the README's examples) over a long prose
   document, even though Section 9b raises the box's limit up to `TEXT_PROMPT_MAX_LEN` characters.

**Interaction with Section 4b (live web search):** if you enabled `ENABLE_WEB_SEARCH` in Section 4b,
the server automatically appends its own tool-use instructions to whatever you type here (or even if
you leave this box empty) -- you do not need to mention search/tools yourself. Those instructions
also consume a small slice of the same character/attention-window budget described below, so if
you're already writing a long, tightly-budgeted grounding prompt, account for a few hundred extra
characters, or set `ENABLE_WEB_SEARCH = False` in Section 4b if you'd rather keep the full budget for
static grounding text only.

In [ ]:
CUSTOM_TOPIC_INFO = (
    "Replace this with your facts, e.g.: Our return policy allows returns within 30 days with a receipt. "
    "Store hours are 9am-6pm Monday-Saturday, closed Sunday."
)
ROLE_SENTENCE = "You are a knowledgeable assistant for [your topic]."

grounded_prompt = (
    f"{ROLE_SENTENCE} Only use the Information below to answer questions; if something is not covered by "
    f"it, say it is outside what you were told and you cannot help with that. Do not use any other "
    f"knowledge. Information: {CUSTOM_TOPIC_INFO}"
)

limit = globals().get("TEXT_PROMPT_MAX_LEN", 1000)
print(f"Characters: {len(grounded_prompt)} (Text Prompt box limit: {limit})")
print()
print(grounded_prompt)
print()
print("Copy the text above into the 'Text Prompt' box in Section 14, before clicking connect.")


## 14. Using the live web UI

Open the URL printed in Section 11 in a browser, allow microphone access when prompted (this is the one
browser permission step that can't be automated), and start talking.

### Voices (from `README.md`)

```
Natural(female): NATF0, NATF1, NATF2, NATF3
Natural(male):   NATM0, NATM1, NATM2, NATM3
Variety(female): VARF0, VARF1, VARF2, VARF3, VARF4
Variety(male):   VARM0, VARM1, VARM2, VARM3, VARM4
```

### Example role prompts (from `README.md`)

- Assistant role: `You are a wise and friendly teacher. Answer questions or provide advice in a clear and
  engaging way.`
- Casual conversation: `You enjoy having a good conversation.`
- Customer service example: `You work for CitySan Services which is a waste management company and your name
  is Ayelen Lucero. Information: Verify customer name Omar Torres. Current schedule: every other week.
  Upcoming pickup: April 12th. Compost bin service available for $8/month add-on.`

See the repo's `README.md` "Prompting Guide" section for more examples and guidance.

### Live web search (tool-calling)

If Section 4b's `ENABLE_WEB_SEARCH` is on, you can also ask things like:

- "What's today's gold price in Bangladesh?"
- "What's the weather like in Dhaka right now?"
- "What's the latest news about a topic you care about?"
- "What's the USD to BDT exchange rate today?"

Helium decides for itself whether a question needs a search -- ordinary questions get an immediate
answer as before. For a live-info question, expect a short **silent pause** (a few seconds, no
filler audio) while it searches and reads the results back to itself, then it answers normally in
its own voice, in the same session. See Section 4b for exactly how this works and its limits.

## 15. GPU optimization notes for RTX 5090

- **bf16 by default**: `moshi/moshi/models/loaders.py:get_moshi_lm` loads the LM in `torch.bfloat16`. Blackwell
  has fast native bf16 Tensor Core throughput, so no dtype changes are needed.
- **`--cpu-offload` is unnecessary here**: it exists in `server.py`/`offline.py` for GPUs with insufficient
  VRAM (via `accelerate`'s `infer_auto_device_map`). An RTX 5090's 32GB comfortably fits this 7B model plus the
  Mimi codec; only enable `USE_CPU_OFFLOAD` if you're also running other large workloads on the same GPU.
- **One process = one GPU's worth of model**: `ServerState.__init__` loads two Mimi instances and the LM once
  per process and keeps them resident (`streaming_forever`). Don't launch a second `moshi.server` process
  against the same GPU unless you've confirmed there's VRAM headroom — check with `nvidia-smi`.
- **Warmup cost is already handled**: `state.warmup()` runs 4 dummy frames through the full encode → LM →
  decode path right after model load, ahead of any real connections — that's the per-process latency you saw
  the log wait for in Section 10, not something to optimize further.
- **CUDA build must match the GPU**: Blackwell (`sm_120`) needs the `cu130` PyTorch wheels installed in
  Section 5. If you ever see `no kernel image is available for execution on the device`, that cell needs to be
  re-run (something likely reinstalled a different torch build afterward).

## 16. Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| `401`/`403` downloading model assets | License not accepted, or `HF_TOKEN` doesn't belong to the account that accepted it | Re-check Section 7: accept the license at the model page with the **same** account whose token you pasted |
| `no kernel image is available for execution on the device` | Torch build doesn't have Blackwell (`sm_120`) kernels | Re-run the `cu130` reinstall cell in Section 5, then re-run Section 6's smoke test |
| `ImportError` / build errors mentioning `opus` while installing `sphn` | `libopus-dev` missing | Re-run Section 3, then re-run Section 5 |
| Server process exits immediately, log shows a CUDA OOM | Not enough free VRAM (e.g. another process holds the GPU) | Check `nvidia-smi`; consider `USE_CPU_OFFLOAD = True` (Section 2) and re-run Sections 10‑11 |
| Browser blocks microphone / `getUserMedia` fails | Page wasn't loaded over a secure context | Use the RunPod **proxy** URL from Section 11 (HTTPS at the edge), not a plain `http://<pod-ip>:8998` URL |
| Can't reach the URL from outside the pod at all | Port not exposed | In the RunPod console, add `SERVER_PORT` as an HTTP Service on the pod's Connect page |
| First launch seems to hang for several minutes | Normal — first run downloads multi-GB weights and extracts `voices.tgz`/`dist.tgz` | Watch the log tail printed by Section 10; increase `TIMEOUT_S` if your network is slow |
| `mkcert` warnings in the log | Only relevant when `USE_APP_TLS = True`; `moshi/moshi/utils/connection.py` falls back to plain HTTP automatically if `mkcert` can't be installed | Safe to ignore in default (proxy) mode |

### Recap: what genuinely cannot be automated by this notebook
1. Clicking "Agree and access repository" on the gated HF model page.
2. Issuing the HF access token for that account.
3. Exposing `SERVER_PORT` as an HTTP Service in the RunPod console.
4. The browser's microphone-permission prompt.

## 17. Stop the server (run only when you want to shut it down)

This is a management utility cell, **not** part of the linear startup flow — running it will terminate the
backend you just verified above. Run it deliberately when you're done with the session, not as part of a
top-to-bottom "Run All".

In [ ]:
# Intentionally NOT meant to run automatically as part of the startup sequence above.
try:
    server_proc.terminate()
    server_proc.wait(timeout=15)
    print(f"Server process {server_proc.pid} stopped.")
except NameError:
    print("No server_proc in scope -- nothing to stop.")
except subprocess.TimeoutExpired:
    server_proc.kill()
    print(f"Server process {server_proc.pid} killed after not stopping gracefully.")